# Week 7 Assignment: Price Band Classification with Open-Source Models

## Task
Fine-tune an open-source LLM to predict price class (budget, midrange, premium) instead of exact price.

## Objectives:
1. Load and prepare ed-donner dataset
2. Create price bands (budget, midrange, premium) based on data distribution
3. Fine-tune Llama-3.2-3B using QLoRA for classification
4. Evaluate with F1 score, accuracy, and classification metrics

## 1. Installation and Imports

In [ ]:
# Install required packages
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!pip install -q --upgrade transformers datasets peft wandb scikit-learn

In [ ]:
# Core imports
import os
import re
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime
from collections import Counter

# Google Colab (comment out if running locally)
from google.colab import userdata

# HuggingFace and Model imports
from huggingface_hub import login
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Metrics
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

# Optional: Weights & Biases for tracking
import wandb

print("All imports successful")

## 2. Configuration

In [ ]:
# ============ Model Configuration ============
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price-band-classification"
HF_USER = "sirmuelemmanuel"  # Update this!

# ============ Data Configuration ============
LITE_MODE = True  # Set to False for full dataset
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# ============ Training Configuration ============
EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 16 if LITE_MODE else 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# ============ QLoRA Configuration ============
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 64
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS

# ============ Optimizer Configuration ============
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

# ============ Logging Configuration ============
VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True  # Set to True if you want to use W&B

# ============ Run Name ============
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ============ Price Band Thresholds ============
# These will be calculated from the data
BUDGET_MAX = None
MIDRANGE_MAX = None

# ============ Random Seed ============
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

# Check GPU capability for bf16
capability = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = capability[0] >= 8

print(f"🚀 Configuration Complete")
print(f"Model: {BASE_MODEL}")
print(f"📁 Dataset: {DATASET_NAME}")
print(f"🔧 Mode: {'LITE' if LITE_MODE else 'FULL'}")
print(f"🎯 GPU bf16 support: {use_bf16}")
print(f"Run name: {RUN_NAME}")

## 3. Authentication

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)
hf_token = userdata.get('HF_TOKEN')

# If running on Colab, uncomment:
# hf_token = userdata.get('HF_TOKEN')

login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace")

In [ ]:
# Optional: Login to Weights & Biases
if LOG_TO_WANDB:
    # For Colab: wandb_api_key = userdata.get('WANDB_API_KEY')
    wandb_api_key = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login()

    os.environ["WANDB_PROJECT"] = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"] = "false"
    print("Logged in to Weights & Biases")
else:
    print("⏭️  Skipping W&B login")

## 4. Load and Analyze Data

In [ ]:
# Load the dataset from HuggingFace
dataset = load_dataset(DATASET_NAME)

# Extract train, validation, and test sets
train_data = dataset['train']
val_data = dataset['val']
test_data = dataset['test']

print(f"Dataset loaded successfully")
print(f"   Training samples: {len(train_data):,}")
print(f"   Validation samples: {len(val_data):,}")
print(f"   Test samples: {len(test_data):,}")
print(f"\nSample data point:")
print(f"   Prompt: {train_data[0]['prompt'][:200]}...")
print(f"   Completion: {train_data[0]['completion']}")

In [ ]:
# Extract all prices to analyze distribution
def extract_price(completion_str):
    """Extract numeric price from completion string"""
    try:
        price_str = str(completion_str).replace('$', '').replace(',', '').strip()
        return float(price_str)
    except:
        return None

# Collect all prices from all splits
all_prices = []
for split in [train_data, val_data, test_data]:
    for item in split:
        price = extract_price(item['completion'])
        if price is not None:
            all_prices.append(price)

all_prices = np.array(all_prices)

print("💰 Price Distribution Analysis:")
print(f"   Total items: {len(all_prices):,}")
print(f"   Min price: ${np.min(all_prices):.2f}")
print(f"   Max price: ${np.max(all_prices):.2f}")
print(f"   Mean price: ${np.mean(all_prices):.2f}")
print(f"   Median price: ${np.median(all_prices):.2f}")
print(f"   25th percentile: ${np.percentile(all_prices, 25):.2f}")
print(f"   75th percentile: ${np.percentile(all_prices, 75):.2f}")

## 5. Define Price Bands

In [ ]:
# Define price bands using percentiles for balanced classes
# Using 33rd and 67th percentiles gives roughly equal distribution

BUDGET_MAX = np.percentile(all_prices, 33.33)
MIDRANGE_MAX = np.percentile(all_prices, 66.67)

print("🏷️  Price Band Definitions:")
print(f"   💵 Budget: $0 - ${BUDGET_MAX:.2f}")
print(f"   💰 Midrange: ${BUDGET_MAX:.2f} - ${MIDRANGE_MAX:.2f}")
print(f"   💎 Premium: ${MIDRANGE_MAX:.2f}+")

def get_price_band(price):
    """Classify a price into budget, midrange, or premium"""
    if price <= BUDGET_MAX:
        return "budget"
    elif price <= MIDRANGE_MAX:
        return "midrange"
    else:
        return "premium"

# Test the classification
test_prices = [10, 50, 100, 200, 500]
print("\nTest classifications:")
for price in test_prices:
    band = get_price_band(price)
    print(f"   ${price:.2f} → {band}")

# Analyze distribution
bands = [get_price_band(p) for p in all_prices]
band_counts = Counter(bands)

print("\nClass Distribution:")
for band in ['budget', 'midrange', 'premium']:
    count = band_counts[band]
    percentage = (count / len(bands)) * 100
    print(f"   {band.capitalize()}: {count:,} ({percentage:.1f}%)")

## 6. Prepare Classification Dataset

In [ ]:
def transform_to_classification(example):
    """Transform price prediction example to classification example"""
    price = extract_price(example['completion'])
    if price is None:
        return None

    band = get_price_band(price)

    # Create new prompt for classification
    # We keep the original prompt but change the completion to the class
    return {
        'prompt': example['prompt'],
        'completion': band,
        'original_price': price
    }

# Transform all datasets
print("Transforming datasets to classification format...")

train_classification = []
for item in tqdm(train_data, desc="Processing train"):
    transformed = transform_to_classification(item)
    if transformed:
        train_classification.append(transformed)

val_classification = []
for item in tqdm(val_data, desc="Processing validation"):
    transformed = transform_to_classification(item)
    if transformed:
        val_classification.append(transformed)

test_classification = []
for item in tqdm(test_data, desc="Processing test"):
    transformed = transform_to_classification(item)
    if transformed:
        test_classification.append(transformed)

# Convert to HuggingFace datasets
train_dataset = Dataset.from_list(train_classification)
val_dataset = Dataset.from_list(val_classification[:VAL_SIZE])
test_dataset = Dataset.from_list(test_classification)

print(f"\nDatasets transformed")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val: {len(val_dataset):,} samples")
print(f"   Test: {len(test_dataset):,} samples")
print(f"\nSample transformed data:")
print(f"   Prompt: {train_dataset[0]['prompt'][:150]}...")
print(f"   Original Price: ${train_dataset[0]['original_price']:.2f}")
print(f"   Class: {train_dataset[0]['completion']}")

## 7. Format Prompts for Training

In [ ]:
def format_prompt_for_training(example):
    """Format examples in chat format for Llama models"""
    # Modify the prompt to ask for classification
    classification_prompt = example['prompt'].replace(
        "What is the price?",
        "What is the price category? Respond with exactly one word: budget, midrange, or premium."
    )

    # If the prompt doesn't contain "What is the price?", add the instruction
    if "What is the price category?" not in classification_prompt:
        classification_prompt += "\n\nWhat is the price category? Respond with exactly one word: budget, midrange, or premium."

    text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{classification_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['completion']}<|eot_id|>"

    return {'text': text}

# Apply formatting
train_dataset = train_dataset.map(format_prompt_for_training)
val_dataset = val_dataset.map(format_prompt_for_training)
test_dataset = test_dataset.map(format_prompt_for_training)

print("Prompts formatted for training")
print(f"\nSample formatted text:")
print(train_dataset[0]['text'][:500] + "...")

## 8. Load Model and Tokenizer

In [ ]:
# Configure quantization
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16, # if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16, # if use_bf16 else torch.float16,
    )

print(f"⚙️  Quantization configured: {'4-bit' if QUANT_4_BIT else '8-bit'}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Prepare model for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

print(f"Model loaded")
print(f"💾 Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

## 9. Configure LoRA and Training Parameters

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

print("LoRA configuration created")
print(f"   r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"   Target modules: {TARGET_MODULES}")

In [ ]:
# Training configuration
training_config = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else "none",
    push_to_hub=False,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("Training configuration created")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Precision: {'bf16' if use_bf16 else 'fp16'}")

## 10. Initialize Trainer and Start Training

In [ ]:
# Initialize W&B run if enabled
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)
    print("W&B run initialized")

In [ ]:
# Create the SFT Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_config,
)

print("Trainer initialized")
print("🚀 Ready to train!")

In [ ]:
# Start training
print("🏋️  Starting training...\n")
trainer.train()
print("\nTraining complete!")

## 11. Save the Fine-tuned Model

In [ ]:
# Save the model locally
trainer.model.save_pretrained(PROJECT_RUN_NAME)
tokenizer.save_pretrained(PROJECT_RUN_NAME)

print(f"Model saved to {PROJECT_RUN_NAME}")

# Optional: Push to HuggingFace Hub
# Uncomment the following lines if you want to push to the hub
trainer.model.push_to_hub(HUB_MODEL_NAME)
tokenizer.push_to_hub(HUB_MODEL_NAME)
print(f"Model pushed to HuggingFace Hub: {HUB_MODEL_NAME}")

## 12. Evaluation

In [ ]:
# Load the fine-tuned model for evaluation
print("📥 Loading fine-tuned model for evaluation...")

# Load base model again
eval_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

# Load the LoRA adapter
eval_model = PeftModel.from_pretrained(eval_base_model, PROJECT_RUN_NAME)
eval_model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Fine-tuned model loaded for evaluation")

In [ ]:
def predict_price_band(model, tokenizer, prompt, max_new_tokens=10):
    """Predict price band for a given prompt"""
    # Format the prompt for classification
    classification_prompt = prompt.replace(
        "What is the price?",
        "What is the price category? Respond with exactly one word: budget, midrange, or premium."
    )

    if "What is the price category?" not in classification_prompt:
        classification_prompt += "\n\nWhat is the price category? Respond with exactly one word: budget, midrange, or premium."

    # Format in chat format
    formatted_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{classification_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the prediction (last part after assistant tag)
    if "assistant" in response:
        prediction = response.split("assistant")[-1].strip()
    else:
        prediction = response[len(formatted_prompt):].strip()

    # Clean up and extract just the class (FIXED LOGIC)
    valid_labels = ['budget', 'midrange', 'premium']
    prediction_lower = prediction.lower()

    # First, try to find an exact whole-word match using regex
    match = re.search(r'\b(' + '|'.join(valid_labels) + r')\b', prediction_lower)
    if match:
        return match.group(1)

    # If no whole-word match (e.g., due to concatenation like 'midrangebudgetpremium'),
    # search for any of the valid labels as a substring. The order in `valid_labels`
    # will determine which label is returned if multiple are found concatenated.
    for label in valid_labels:
        if label in prediction_lower:
            return label

    # If no valid label is found at all (which shouldn't happen if the model tries to respond),
    # return a default valid label to prevent errors in metrics calculation.
    return 'midrange'


In [ ]:
# Test on a few samples
print("Testing on sample predictions:\n")

for i in range(15, 20):
    sample = test_dataset[i]
    prediction = predict_price_band(eval_model, tokenizer, sample['prompt'])
    actual = sample['completion']
    price = sample['original_price']

    status = "✅" if prediction == actual else "❌"
    print(f"{status} Sample {i+1}:")
    print(f"   Price: ${price:.2f}")
    print(f"   Actual: {actual}")
    print(f"   Predicted: {prediction}")
    print()

In [ ]:
# Full evaluation on test set
print("Running full evaluation on test set...\n")

# Use a subset for faster evaluation in LITE mode
eval_size = 500 if LITE_MODE else len(test_dataset)
test_subset = test_dataset.select(range(min(eval_size, len(test_dataset))))

predictions = []
actuals = []

for i, sample in enumerate(tqdm(test_subset, desc="Evaluating")):
    prediction = predict_price_band(eval_model, tokenizer, sample['prompt'])
    predictions.append(prediction)
    actuals.append(sample['completion'])

print(f"\nEvaluation complete on {len(predictions)} samples")

In [ ]:
# Calculate metrics
accuracy = accuracy_score(actuals, predictions)
f1_macro = f1_score(actuals, predictions, average='macro')
f1_weighted = f1_score(actuals, predictions, average='weighted')
precision_macro = precision_score(actuals, predictions, average='macro', zero_division=0)
recall_macro = recall_score(actuals, predictions, average='macro', zero_division=0)

print("\n" + "="*60)
print("📈 CLASSIFICATION METRICS")
print("="*60)
print(f"\n🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")
print(f"🎪 Precision (Macro): {precision_macro:.4f}")
print(f"🔍 Recall (Macro): {recall_macro:.4f}")

# Classification report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
# Using labels parameter to ensure consistency even if some classes are missing from predictions
print("\n" + classification_report(actuals, predictions,
                                    labels=['budget', 'midrange', 'premium'],
                                    target_names=['budget', 'midrange', 'premium'],
                                    zero_division=0))

# Confusion matrix
cm = confusion_matrix(actuals, predictions, labels=['budget', 'midrange', 'premium'])
print("\n" + "="*60)
print("🔢 CONFUSION MATRIX")
print("="*60)
print("\nRows: Actual | Columns: Predicted")
print("\n              Budget    Midrange  Premium")
print(f"Budget        {cm[0][0]:6d}    {cm[0][1]:8d}  {cm[0][2]:7d}")
print(f"Midrange      {cm[1][0]:6d}    {cm[1][1]:8d}  {cm[1][2]:7d}")
print(f"Premium       {cm[2][0]:6d}    {cm[2][1]:8d}  {cm[2][2]:7d}")
print("\n" + "="*60)

## 13. Error Analysis

In [ ]:
# Analyze errors
errors = []
for i, (pred, actual) in enumerate(zip(predictions, actuals)):
    if pred != actual:
        errors.append({
            'index': i,
            'price': test_subset[i]['original_price'],
            'actual': actual,
            'predicted': pred,
            'prompt': test_subset[i]['prompt'][:150]
        })

print(f"\n🔍 Error Analysis:")
print(f"   Total errors: {len(errors)} out of {len(predictions)} ({len(errors)/len(predictions)*100:.2f}%)")

if errors:
    print(f"\nSample errors (first 10):\n")
    for i, error in enumerate(errors[:10]):
        print(f"{i+1}. Price: ${error['price']:.2f}")
        print(f"   Actual: {error['actual']} | Predicted: {error['predicted']}")
        print(f"   Prompt: {error['prompt']}...")
        print()